# Chapter 03 — If There's Any Doubt, It's Deterministic

**Companion to Applied AI**

Question: Can we demonstrate why deterministic work should not be routed through a model?

By the end of this notebook you will have:

- classified pipeline operations as deterministic or stochastic
- shown deterministic outputs repeating identically
- measured the cost/variance penalty of misrouting deterministic work

## What this notebook demonstrates
A synthetic pipeline with six operations. Five are deterministic; one genuinely needs sampling. A seeded mock stands in for the model.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import hashlib, re

seed: 42


## 1. The pipeline, classified up front

In [2]:
OPS = {
    "hash_compare":      "deterministic",
    "budget_calculation":"deterministic",
    "span_extraction":   "deterministic",
    "model_judgment":    "stochastic",
    "source_validation": "deterministic",
    "policy_decision":   "deterministic",
}
for k, v in OPS.items():
    print(f"{k:20s} -> {v}")
assert sum(1 for v in OPS.values() if v == "stochastic") == 1

hash_compare         -> deterministic
budget_calculation   -> deterministic
span_extraction      -> deterministic
model_judgment       -> stochastic
source_validation    -> deterministic
policy_decision      -> deterministic


## 2. Deterministic operations repeat identically

In [3]:
def hash_compare(a: bytes, b: bytes) -> bool:
    return hashlib.sha256(a).hexdigest() == hashlib.sha256(b).hexdigest()

def budget_calculation(input_tokens: int, output_tokens: int) -> float:
    return (input_tokens / 1e6) * 2.50 + (output_tokens / 1e6) * 10.00

def span_extraction(text: str):
    return re.findall(r"E_[A-Z_]+", text)

for _ in range(3):
    assert hash_compare(b"abc", b"abc") is True
    assert budget_calculation(1000, 500) == budget_calculation(1000, 500)
    assert span_extraction("row 14 E_OVERLIMIT, row 31 E_MISSING_REF") == ["E_OVERLIMIT", "E_MISSING_REF"]
print("deterministic ops: 3/3 identical repeats")

deterministic ops: 3/3 identical repeats


## 3. The one stochastic step varies by design

In [4]:
def mock_judgment(case: str, trial: int) -> str:
    """Seeded stand-in for a sampled judgment: varies across trials."""
    r = random.Random(hash((SEED, case, trial)) % (2**32))
    return "needs_fix" if r.random() < 0.7 else "ok"

outcomes = [mock_judgment("case-A", t) for t in range(10)]
print(outcomes)
assert len(set(outcomes)) > 1, "expected variation across trials"

['needs_fix', 'needs_fix', 'ok', 'needs_fix', 'needs_fix', 'needs_fix', 'ok', 'needs_fix', 'needs_fix', 'needs_fix']


## 4. The penalty for misrouting: cost + variance on work that needed neither

In [5]:
N = 1000
det_cost = N * 0.0            # local deterministic span extraction
misrouted_cost = N * 0.002    # $0.002 per mock model call
print(f"deterministic path cost: ${det_cost:.2f}")
print(f"misrouted path cost:     ${misrouted_cost:.2f}")
print(f"deterministic variance across reruns: 0 outcomes differ")
mis = [mock_judgment("span-task", t) for t in range(20)]
print(f"misrouted disagreement across 20 reruns: {sum(1 for m in mis if m != mis[0])} differ from first")
assert misrouted_cost > det_cost

deterministic path cost: $0.00
misrouted path cost:     $2.00
deterministic variance across reruns: 0 outcomes differ
misrouted disagreement across 20 reruns: 4 differ from first


## Interpretation
- Supports: deterministic steps are free, exact, and repeatable; routing them through sampling adds cost and variance for no capability gain.
- Does NOT support: a universal rule for where the boundary sits; classification is per-operation work.

## Try it yourself
1. Move `span_extraction` to stochastic and rerun section 4 with your own per-call price.
2. Change the 0.7 base rate and watch disagreement grow or shrink.
3. Add a `source_validation` check that refuses model-parsed spans unless they match the regex.